# Keystone Assessments — Dataset Documentation

**Dataset:** `keystone_assessments.csv`

**Purpose:** Tracks student Keystone assessment administrations, scaled scores, testing attempts, and assessment dates within the synthetic Graduation Tracker student information system. Supports analysis of student assessment performance, identification of students requiring additional testing or interventions, and monitoring of assessment-related graduation readiness.

**Record level:** One row per student, assessment subject, and testing attempt.

**Primary key:** `assessment_id`

**Foreign key:** `student_id` → `students.csv`

**Fields:**

- `assessment_id`: Unique, randomly generated 10-digit assessment administration identifier, stored as a string to preserve leading zeros.
- `student_id`: Student identifier linking assessment records to the student dataset.
- `assessment_subject`: Keystone assessment subject (Algebra 1, Biology, or Literature).
- `attempt_number`: Number indicating the student's testing attempt for the assessment subject (1 or 2).
- `scaled_score`: Numeric assessment score ranging from 1200 to 1800.
- `assessment_season`: Testing administration season (Spring, Summer, or Winter).
- `assessment_year`: Calendar year of the assessment administration (2024, 2025, or 2026), stored as a string.

**Business rules:**

- Each student is independently evaluated for participation in each of the three Keystone assessment subjects:
  - Algebra 1
  - Biology
  - Literature
- Each student has a 75% probability of receiving at least one assessment record for each subject.
- Students may have no assessment records for one or more subjects.
- Students selected for an assessment receive either one or two testing attempts:
  - One attempt: 85% probability.
  - Two attempts: 15% probability.
- Assessment scores are randomly generated using a normal distribution with a mean of 1480 and standard deviation of 95.
- Generated scores are converted to integers and restricted to the range of 1200–1800.
- Assessment seasons follow these probability distributions:
  - Spring: 85%
  - Summer: 5%
  - Winter: 10%
- Assessment years are randomly selected from 2024, 2025, and 2026 with equal probability.
- Assessment IDs are randomly generated as unique 10-digit strings.
- Each assessment administration must have a unique `assessment_id`.
- Every `student_id` must exist in the student dataset.
- All scaled scores must fall between 1200 and 1800.
- Random generation uses a fixed seed of 42 to support reproducibility.

**Data validation:**

- Verify all assessment IDs are unique.
- Verify all student IDs exist in the student dataset.
- Verify all scaled scores fall between 1200 and 1800.
- Generate a count of assessment records by assessment subject.

**Data source:** `students.csv` provides the student identifiers used to generate Keystone assessment records.

**Generation method:** Synthetic Keystone assessment administration records generated using Python, pandas, and NumPy random sampling.

**Limitations:** Assessment participation, scaled scores, testing seasons, and assessment years are synthetically generated and do not represent actual Pennsylvania Keystone assessment results. Testing attempts are generated independently without enforcing chronological order, meaning a second attempt may be assigned an earlier testing year or season than the first attempt. Assessment participation and performance are not linked to student grade levels, enrollment histories, course completion, or graduation cohorts. The dataset does not explicitly classify assessment performance levels or determine whether students satisfy Pennsylvania Act 158 graduation pathway requirements. Assessment records are intended for portfolio demonstration and analytical development rather than official student accountability reporting.

In [ ]:
# Import libraries

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive

In [ ]:
# Connect Google Drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Create folder path

folder_path = Path('/content/drive/MyDrive/Graduation_Tracker/data/ground_truth')

In [ ]:
# Set random seed

rng = np.random.default_rng(42)

In [ ]:
# Create simulated assessment data

students = pd.read_csv(folder_path/'students.csv')

subjects = ['Algebra 1', 'Biology', 'Literature']

records = []

for id in students.student_id:
  for subject in subjects:
    if rng.random() < 0.75:
      for attempt in range(1, int(rng.choice([1,2], p = [0.85, 0.15])) + 1):
        score = int(np.clip(rng.normal(1480, 95), 1200, 1800))
        season = rng.choice(['Spring', 'Summer', 'Winter'], p= [0.85, 0.05, 0.10])
        records.append({
            'student_id':id,
            'assessment_subject':subject,
            'attempt_number':attempt,
            'scaled_score':score,
            'assessment_season':season,
            'assessment_year':f'{2024 + int(rng.integers(0,3))}'
        })

keystones = pd.DataFrame(records)

assessment_ids = set()

while len(assessment_ids) < len(keystones):
    assessment_ids.add(f'{rng.integers(0, 10**10):010d}')

keystones.insert(0, 'assessment_id', list(assessment_ids))

display(keystones.head(10))

,assessment_id,student_id,assessment_subject,attempt_number,scaled_score,assessment_season,assessment_year
0,4904914828,1145269,Biology,1,1569,Spring,2025
1,4988732179,1145269,Biology,2,1492,Spring,2026
2,5504034372,1145269,Literature,1,1563,Winter,2026
3,2775552963,1145270,Biology,1,1515,Spring,2025
4,9886788290,1145271,Algebra 1,1,1415,Winter,2025
5,8791364885,1145271,Literature,1,1514,Spring,2026
6,1400877484,1145272,Algebra 1,1,1441,Spring,2026
7,7146658052,1145272,Biology,1,1469,Spring,2025
8,9115624569,1145272,Literature,1,1550,Spring,2025
9,1336640257,1145273,Algebra 1,1,1502,Spring,2026


In [ ]:
# Validate dataset

assert keystones.assessment_id.is_unique

assert keystones.student_id.isin(students.student_id).all()

assert keystones.scaled_score.between(1200, 1800).all()

print(keystones.groupby('assessment_subject').size())

assessment_subject
Algebra 1     4360
Biology       4331
Literature    4284
dtype: int64


In [ ]:
# Save CSV to Google Drive

csv_path = folder_path / 'keystone_assessments.csv'

keystones.to_csv(csv_path, index = False)